# Introduction to Workbench Workbook

This workbook describes the solutions to the problems offered in the [Introduction to Workbench kata](./IntroToWorkbench.ipynb).



In [ ]:
from psiqdk.workbench import QPU, Qubits, Qubrick
import numpy as np
from typing import Callable

## Problem 1. Instantiate a QPU

All we need to do is pass the input argument `num_qubits` to the `num_qubits` parameter of a `QPU`. This creates a QPU with an upper bound of `num_qubits` qubits.

In [ ]:
def instantiate_qpu(num_qubits: int):
    qpu = QPU(num_qubits=num_qubits)
    return qpu

## Problem 2. Instantiate a qubit register

All we need to do is pass the input argument `num_qubits` to the first parameter of the `Qubits` object, the `name` to the second parameter, and the `qpu` to the third parameter. This will create a `Qubits` object that is linked to the given QPU with the given size and given name.

In [ ]:
def instantiate_qubits(qpu: QPU, num_qubits: int, name: str):
    qubits = Qubits(num_qubits, name, qpu)
    return qubits

## Problem 3. Apply quantum gates

Since we are given a `Qubits` object and can apply gates directly to it, we simply perform the two gates in order. First, we call the method `x()` of the `Qubits` object;  this applies the `x` gate. Then we do the same with the `rx()` method, passing the given `angle` parameter to apply the specific rotation angle.

In [ ]:
def apply_gates(reg: Qubits, angle: int):
    reg.x()
    reg.rx(angle)

## Problem 4. Apply controlled quantum gates

Since the first two controlled gates use `reg2` as the target, we'll call the `x` method on `reg2` twice. However, in the first call we only use `reg1` as the condition parameter, since all qubits being in the $\ket{1}$ state is the default control condition. In the second call, we use `reg1==0` to condition on the $\ket{0}$ state of `reg1` instead.

In the third gate, we call an `x` gate on `reg1` and use `reg2` as our condition.

In [ ]:
def apply_controlled_gates(reg1: Qubits, reg2: Qubits):
    reg2.x(cond=reg1)     # Apply a controlled-x gate to reg2, controlled on reg1 = 1
    reg2.x(cond=reg1==0)  # Apply a controlled-x gate to reg2, controlled on reg1 = 0
    reg1.x(cond=reg2)     # Apply a controlled-x gate to reg1, controlled on reg2 = 1

## Problem 5. Indexing and slicing

For the first gate, we need to apply an `x` gate to only one qubit, the second qubit in the register.  We can do this by directly indexing the register with index $1$.

For the second gate, we need to apply an `x` gate to multiple qubits from the register at once.  We exclude the qubit with index $0$, since the task tells us to ignore it. We use a start qubit from the next even-indexed qubit at index $2$. Then we use a step of $2$ to capture every other even-indexed qubit in the register.

In [ ]:
def indexing_and_slicing(reg: Qubits):
    reg[1].x()       # Apply an x gate to the second qubit in the register
    reg[2::2].x()    # Apply an x gate to every even qubits except the first with a single operation

## Problem 6. Combining registers

Since every concatenation returns a new `Qubits` object, we simply need to use the `|` operator between the requested sets, then apply the corresponding gate to the resulting object.

In [ ]:
def concatenate_registers(reg1: Qubits, reg2: Qubits, reg3: Qubits):
    (reg1 | reg3).x()        # Apply an x gate to the concatenation of the first and third registers
    (reg2 | reg3).y()        # Apply an y gate to the concatenation of the second and third registers
    (reg1 | reg2 | reg3).z() # Apply an z gate to the concatenation of the first, second and third registers

## Problem 7. Using the state vector

In this problem we need to examine each component of the state vector. We first get the state vector by using `qpu.pull_state()`.

Next, we find the maximum element of the state vector array using Python function `max`. Finally, we return the list of all indices for which the corresponding element of the array is close to the maximum using list comprehension combined with `enumerate`.

In [ ]:
def examine_state_vector(qpu: QPU) -> list[int]:
    state_vector = qpu.pull_state()
    max_amp = max(state_vector)
    return [index for index, amp in enumerate(state_vector) if np.allclose(amp, max_amp)]

## Problem 8. Using measurements

We use a for loop to run the function $1000$ times. On each iteration, we use the function we are given to get a `Qubits` object and then use its `read()` method to get a measurement result from it. In this case, the prepared state is a superposition of the all $\ket{0}$ and the all $\ket{1}$ basis states.

Then, we check if the measurement result exists in the dictionary. If it does not, we add it with read count $0$. Then we increment the dictionary counter for the measurement result.

Finally, we return the dictionary.

In [ ]:
def use_measurements(func: Callable[[], Qubits]) -> dict[int, int]:
    result_dict = {}
    for _ in range(1000):
        reg = func()
        res = reg.read()
        if res not in result_dict:
            result_dict[res] = 0
        result_dict[res] += 1
    
    return result_dict

## Problem 9. Build a Qubrick

For the GHZ Qubrick, we only need to define the `_compute` method that takes a single `Qubits` register. Since we need to put one control qubit into superposition, we apply a Hadamard gate to qubit $0$.  Then to entangle it with the other qubits in the register, we iterate over the qubits at indices $1$ to $n-1$ and apply an `x` gate controlled by qubit $0$ to each one.

In [ ]:
class GHZ(Qubrick):
    def _compute(self, reg: Qubits):
        reg[0].had()
        for r in reg[1:]:
            r.x(cond=reg[0])

## Problem 10. Apply a Qubrick

To apply the `GHZ` Qubrick, we first instantiate it with no arguments, since it has no constructor arguments to set.

Then, we use the `compute` method of the `GHZ` Qubrick, passing the register as the argument.

Finally, we return the `QPU`.

In [ ]:
def apply_qubrick(n: int) -> QPU:
    qpu = QPU(num_qubits=n)
    reg = Qubits(n, "reg", qpu=qpu)
    ghz = GHZ()
    ghz.compute(reg)
    return qpu

> Copyright (c) 2026 PsiQuantum